# Tuning and LLM for Sentiment Analysis

The conda environment set up to run this example is

```shell
conda create --name transformer python=3.10 -y
conda activate transformer
pip install torch transformers datasets accelerate tokenizers numpy matplotlib scikit-learn
```

Each of these libraries contribute the following functionality

- torch: PyTorch for deep learning
- transformers:	Hugging Face model library
- datasets: Load and process datasets
- accelerate: Optimizes model training speed
- tokenizers: Fast tokenization for NLP tasks
- numpy: Array and numerical operations
- matplotlib: Visualization support (optional)
- scikit-learn: Additional evaluation metrics (optional)



In [ ]:
## Fix up the environment for Windows Lab Machine
import os
# Remove invalid SSL override(s)
os.environ.pop("SSL_CERT_FILE", None)
os.environ.pop("SSL_CERT_DIR", None)
import warnings
warnings.filterwarnings("ignore")

import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer
from datasets import load_dataset, concatenate_datasets






## Step 1: Load Pre-trained Model & Tokenizer

This step loads a pre-trained NLP model and tokenizer from the Hugging Face transformers 

`model_name = "distilbert-base-uncased"`

- Defines the model we want to use: DistilBERT.
- "distilbert-base-uncased" is a smaller, faster, and lighter version of BERT.
- "uncased" means it does not differentiate between uppercase and lowercase words (e.g., "Movie" and "movie" are treated the same)

`tokenizer = AutoTokenizer.from_pretrained(model_name)`

- Loads a pre-trained tokenizer for DistilBERT.
- The tokenizer is responsible for:
- Splitting text into tokens (e.g., "This movie was amazing!" → ['this', 'movie', 'was', 'amazing', '!']).
- Mapping tokens to numerical IDs (so the model can process them).
- Adding special tokens (like [CLS] and [SEP] for sentence boundaries).
- Handling padding & truncation (to ensure fixed-length input).

```text
Example:

print(tokenizer("This movie was amazing!"))

{'input_ids': [101, 2023, 3185, 2001, 6429, 999, 102], 
 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

101 → [CLS] (start of the sentence)
2023 → "this"
3185 → "movie"
6429 → "amazing"
999 → "!"
102 → [SEP] (end of the sentence)
attention_mask: Ensures padding tokens are ignored.
```
`model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)`

- Loads a pre-trained DistilBERT model designed for sequence classification.
- The num_labels=2 means the model is being fine-tuned for binary classification (e.g., positive vs. negative sentiment).
- The model processes input text through multiple transformer layers.
- It outputs logits (raw scores) for two labels (Positive = 1, Negative = 0).
- The final prediction is based on the higher logit value.

Before Fine-Tuning:

- The classification layer is randomly initialized (classifier.weight, classifier.bias).
- Hugging Face gives a warning because the model is not yet trained on our dataset.

After Fine-Tuning:

- The classification layer will have learned to predict positive or negative based on IMDb reviews.

  

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)  # Binary classification

## Step 2: Load and Balance the Dataset

This step loads, filters, balances, and prepares the IMDb dataset for fine-tuning.

`dataset = load_dataset("imdb", split="train")`

- Loads the IMDb movie review dataset from Hugging Face’s datasets library.
- The dataset contains:
- 50,000 total reviews
- 25,000 labeled as "positive" (label=1)
- 25,000 labeled as "negative" (label=0)
- We only load the training split (split="train") and exclude the test set.


```text
{
    "text": "This movie was absolutely fantastic. I loved every minute of it!",
    "label": 1  # (1 = Positive, 0 = Negative)
}
```
Since IMDb has more data than we need, we select 1,000 positive and 1,000 negative reviews for training.

```text
dataset.filter(lambda x: x["label"] == 1) → Selects only positive reviews.
dataset.filter(lambda x: x["label"] == 0) → Selects only negative reviews.
shuffle(seed=42) → Shuffles the dataset to ensure random selection.
.select(range(1000)) → Selects 1,000 reviews from each group
concatenate_datasets([positive_samples, negative_samples]) → Combines the two datasets into one.
.shuffle(seed=42) → Ensures a random mix of positive and negative samples
```

Tokenizing the Data for Model Input

Tokenization converts text into model-friendly numerical input.

```text
Inside preprocess_function():
truncation=True → Cuts long reviews to fit max_length=256.
padding="max_length" → Ensures all inputs have the same length.
return_tensors="pt" → Converts the output into PyTorch tensors.
```

Tokenized output

```json
{
    "input_ids": [101, 2023, 3185, 2001, 6429, 999, 102], 
    "attention_mask": [1, 1, 1, 1, 1, 1, 1],
    "label": 1
}
```



In [ ]:
dataset = load_dataset("imdb", split="train")  # Load full train set
print("Loading..")

In [ ]:

# Ensure equal number of positive and negative samples
positive_samples = dataset.filter(lambda x: x["label"] == 1).shuffle(seed=42).select(range(1000))
negative_samples = dataset.filter(lambda x: x["label"] == 0).shuffle(seed=42).select(range(1000))

# Merge and shuffle dataset properly
balanced_dataset = concatenate_datasets([positive_samples, negative_samples]).shuffle(seed=42)

def preprocess_function(examples):
    return tokenizer(
        examples["text"], 
        truncation=True, 
        padding="max_length", 
        max_length=256, 
        return_tensors="pt"  # Ensure tensors are properly formatted
    )

tokenized_dataset = balanced_dataset.map(preprocess_function, batched=True, remove_columns=["text"])

In [ ]:
print("Sample data after tokenization:")
print(tokenized_dataset[:5])


## Step 3: Define Training Arguments

`output_dir="./results`
- Specifies where trained models and checkpoints will be saved.
- The model will be stored in ./results after each epoc

`eval_strategy="epoch"`
- Runs evaluation at the end of each epoch.
- Allows monitoring of the model's performance during training.

`learning_rate=5e-5`
- Sets the learning rate (how fast the model updates weights).
- 5e-5 (or 0.00005) is a common fine-tuning rate for transformers.

`num_train_epochs=3`
- Specifies how many times the model sees the entire dataset.
- More epochs = better learning, but too many can cause overfitting.

`weight_decay=0.01` 
- Prevents overfitting by penalizing large model weights.
- Helps keep the model generalized to new data.

Alternative Values:

```text
0.0:  No regularization (higher risk of overfitting).
0.01: Default, reduces overfitting.
0.1:  Stronger regularization (may hurt performance)
```
`lr_scheduler_type="linear"`
- Uses a learning rate scheduler that gradually reduces the learning rate.
- Helps the model learn quickly at the start and refine slowly late

```text
Alternatives
"linear"	Reduces learning rate gradually (recommended).
"constant"	Keeps the same learning rate throughout training.
"cosine"	Drops learning rate sharply at the end of training.
```



In [ ]:


training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",  # Evaluate after each epoch
    save_strategy="epoch",
    learning_rate=5e-5,  # Increase learning rate slightly
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,  # Increase training epochs
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    lr_scheduler_type="linear",  # Helps adjust learning rate dynamically
)

print("Done")

## Step 4: Train the Model

- Moves the model to GPU (if available) to speed up training.
- Defines the Trainer using training arguments, datasets, and model.
- Starts the fine-tuning process by calling .train().

`Trainer` is a built-in Hugging Face utility for managing fine-tuning.
- Passes the model, training arguments, and datasets.
- Splits the dataset:
- 1,500 samples → Used for training.
- 500 samples → Used for evaluation.
- Automatically manages training, validation, saving, and logging.
- Handles distributed training on multiple GPUs.
- Supports various evaluation strategies.


In [ ]:
# Move Model to GPU Before Training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)  # Move model to GPU for faster training

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset.select(range(1500)),  # Train on 1,500 samples
    eval_dataset=tokenized_dataset.select(range(1500, 2000)),  # Eval on 500 samples
)

trainer.train()


##  Step 5: Save and Load the Model

This step saves the fine-tuned model and tokenizer so they can be reloaded later for inference without needing to retrain.

If you restart the script or want to use the model in a different script, you can reload it like this:

```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Load the trained model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained("./fine_tuned_model")
tokenizer = AutoTokenizer.from_pretrained("./fine_tuned_model")

```

In [ ]:

model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")


## Step 6: Run Inference on New Text

In [ ]:

def classify_text(text):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Detect GPU or fallback to CPU
    model.to(device)  # Move model to the detected device

    inputs = tokenizer(text, return_tensors="pt", max_length=256, truncation=True)
    inputs = {key: value.to(device) for key, value in inputs.items()}  # Move input tensors to the same device

    outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits).item()
    return "Positive" if prediction == 1 else "Negative"

# Test the fine-tuned mode


In [ ]:
sample_text = "This movie was amazing! I loved every moment of it."
print("Sentiment:", classify_text(sample_text))


In [ ]:
# Test Positive Sentences
print("Sentiment:", classify_text("I absolutely loved this film! It was fantastic."))
print("Sentiment:", classify_text("An incredible performance by the cast. A must-watch!"))
print("Sentiment:", classify_text("The storyline was beautiful and touching."))
print("Sentiment:", classify_text("This is one of the best movies I’ve ever seen."))
print("Sentiment:", classify_text("A masterpiece! Brilliant acting and a compelling plot."))

# Test Negative Sentences
print("Sentiment:", classify_text("This was the worst movie I have ever watched."))
print("Sentiment:", classify_text("Terrible acting, bad plot, and a complete waste of time."))
print("Sentiment:", classify_text("I was bored to death, nothing exciting ever happened."))
print("Sentiment:", classify_text("I regret watching this movie. It was awful."))
print("Sentiment:", classify_text("Disappointing in every way. Don't waste your time."))
